# 01 — Data Cleaning & Integration (Deliverable A)

This notebook explores the raw tables, runs the cleaning pipeline in `src/cleaning.py`, and documents A1–A8.

**Clock convention:** all joined timestamps are `Africa/Addis_Ababa` (EAT, UTC+3).

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.cleaning import (
    build_master_tables,
    export_outputs,
    integrity_checks,
    CANONICAL_ZONES,
    parse_mixed_timestamps,
    parse_weather_timestamps,
    normalize_zones_series,
)

RAW = ROOT / 'data' / 'raw'
pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42

## Exploratory look at raw files

In [ ]:
train_raw = pd.read_csv(RAW / 'ride_demand_train.csv')
test_raw = pd.read_csv(RAW / 'ride_demand_test.csv')
weather_raw = pd.read_csv(RAW / 'weather_hourly.csv')
events_raw = pd.read_csv(RAW / 'events_calendar.csv')

for name, df in [('train', train_raw), ('test', test_raw), ('weather', weather_raw), ('events', events_raw)]:
    print(f'=== {name}: {df.shape} ===')
    print(df.dtypes)
    print('nulls:', df.isna().sum().to_dict())
    print()

In [ ]:
print('Raw zone labels (train):', train_raw['zone'].nunique())
print(sorted(train_raw['zone'].unique())[:25], '...')
print('\nSample pickup_hour formats:')
print(train_raw['pickup_hour'].head(10).tolist())
print('\nWeather timestamp samples:')
print(weather_raw['timestamp'].head(3).tolist(), weather_raw['timestamp'].tail(3).tolist())
print('\ntrips describe:')
print(train_raw['trips'].describe())

## A2 — Time & key standardization (timezone proof)

Trip rows with explicit offsets use `+03:00` (EAT). Weather rows mostly end in `Z` (UTC).
If we join without converting, rain would be shifted ~3 hours relative to demand.

In [ ]:
# Evidence: weather Z timestamps vs trip +03:00
w_z = weather_raw['timestamp'].astype(str).str.endswith('Z').mean()
t_eat = train_raw['pickup_hour'].astype(str).str.contains(r'\+03:00', regex=True).mean()
print(f'Fraction weather timestamps ending in Z (UTC): {w_z:.3f}')
print(f'Fraction train pickup_hour with +03:00: {t_eat:.3f}')

w_parsed_utc_as_eat = parse_weather_timestamps(weather_raw['timestamp'])
tmp = weather_raw.copy()
tmp['ts_eat'] = w_parsed_utc_as_eat
tmp['hour_eat'] = tmp['ts_eat'].dt.hour
# Temperature should peak mid-afternoon local time (~14-16 EAT), not morning
curve = tmp.groupby('hour_eat')['temp_c'].mean()
print('Hour (EAT) of mean temp peak:', int(curve.idxmax()), 'C=', round(curve.max(), 2))

fig, ax = plt.subplots(figsize=(8, 4))
curve.plot(ax=ax, marker='o')
ax.set_title('Mean temperature by hour after UTC→EAT conversion')
ax.set_xlabel('Hour (Africa/Addis_Ababa)')
ax.set_ylabel('temp_c')
plt.tight_layout()
plt.show()

In [ ]:
print('Zones BEFORE cleaning (train nunique):', train_raw['zone'].nunique())
print('Zones AFTER normalize map:', normalize_zones_series(train_raw['zone']).nunique())
print('Canonical target list:', CANONICAL_ZONES)

## A1 / A3 / A4 / A5 / A6 / A7 / A8 — Run full pipeline

In [ ]:
bundle = build_master_tables()
export_outputs(bundle)

print('Cleaning log (A1):')
display(bundle['cleaning_log'])

print('Weather join audit (A4):', bundle['weather_audit'])
print('Events join audit (A4):', bundle['events_audit'])
print('Zones after:', bundle['zone_after'])

### A3 — Join map (text diagram)

```
ride_demand_*  (left, grain = zone × hour)
      |
      | many-to-one on pickup_hour == weather.timestamp (EAT)
      v
weather_hourly (unique hour after dedupe)
      |
      | interval join on zone + hour ∈ [event.start-pre, event.end+post]
      v
events_calendar (confirmed only; citywide expanded to 12 zones)
      |
      v
master_train / master_test + engineered features
```

**Why trips are left:** we must keep every zone-hour we need to score; weather/events enrich them.
**Event window:** football/concert ±2h; holidays exact day; confirmed status only.

### A5 — Join proof (3 zone-hours)

In [ ]:
mt = bundle['master_train']

rain_ex = mt[mt['rain_mm'] > 2].sort_values('rain_mm', ascending=False).head(1)
event_ex = mt[mt['is_football_window'] == 1].head(1)
hol_ex = mt[mt['is_public_holiday'] == 1].head(1)

for label, ex in [('rain', rain_ex), ('football', event_ex), ('holiday', hol_ex)]:
    print('---', label, '---')
    cols = ['zone', 'pickup_hour', 'trips', 'temp_c', 'rain_mm', 'in_event_window',
            'is_football_window', 'is_public_holiday', 'event_attendance_nearby', 'rain_class']
    display(ex[cols])

### A6 — Feature engineering table

In [ ]:
display(bundle['data_dictionary'])
print('Model features (' + str(len(bundle['model_features'])) + '):')
print(bundle['model_features'])

### A7 — Integrity checks

In [ ]:
checks = integrity_checks(bundle['master_train'], bundle['master_test'])
for name, ok, detail in checks:
    print(('PASS' if ok else 'FAIL') + f': {name} — {detail}')
assert all(ok for _, ok, _ in checks), 'Some integrity checks failed'

### A8 — Master exports

Written to `data/processed/`:
- `master_train.csv`
- `master_test.csv`
- `data_dictionary_master.csv`
- `cleaning_log.csv`

In [ ]:
print(bundle['master_train'].shape, bundle['master_test'].shape)
bundle['master_train'].head()

## Quick EDA on cleaned demand (feeds Deliverable B)

In [ ]:
clean = bundle['master_train'].dropna(subset=['trips'])
zone_share = clean.groupby('zone')['trips'].sum().sort_values(ascending=False)
zone_share = zone_share / zone_share.sum()
print('Zone share of city trips:')
print((zone_share * 100).round(1))

hourly = clean.groupby('hour')['trips'].mean()
fig, ax = plt.subplots(figsize=(8, 4))
hourly.plot(ax=ax, marker='o')
ax.set_title('City-wide mean trips by hour (cleaned)')
ax.set_xlabel('Hour (EAT)')
ax.set_ylabel('Mean trips')
plt.tight_layout()
plt.show()